# Quy trình Tiền Xử lý Dữ liệu – Bộ dữ liệu [P] (Electric Vehicle Population, Bang Washington)

**Môn học:** Phân tích Dữ liệu Kinh doanh (IS403)  
**Đề tài:** Phân tích đặc điểm, xu hướng phân bố và dự báo số lượng ô tô điện (BEV và PHEV) tại Washington, Hoa Kỳ  
**Báo cáo tham chiếu:** Nhóm 8 – Trần Nhụy Tam Tử Phục  

---

### Thông tin tập dữ liệu
- **Dữ liệu đầu vào:** `data/EV_Population_Washington_Clean__1_.csv` (134.726 dòng × 11 thuộc tính)
- **Dữ liệu đầu ra:** `data/processed/EV_Population_WA_Processed.csv` (134.726 dòng × 15 thuộc tính) và `results/P_preprocessing_audit.csv`

### Các nguyên tắc cốt lõi thừa hưởng từ Báo cáo (Mục 3.2):
1. **Dữ liệu cắt ngang (Cross-sectional):** Phản ánh ảnh chụp hiện trạng đăng ký tại một thời điểm (không phải chuỗi thời gian).
2. **Không thay thế giá trị thiếu bằng 0:** Giữ nguyên trạng thái `NA` cho `Electric Range` và `Base MSRP`; các phép kiểm định chỉ tính trên quan sát hợp lệ (valid cases).
3. **Không tùy tiện xóa dòng lặp:** Do file sạch không còn cột mã định danh `DOL Vehicle ID`, các dòng có cùng thuộc tính đại diện cho các phương tiện có thật cùng loại trong cùng khu vực; chỉ đếm và ghi nhận.
4. **Bảo toàn tên cột gốc:** Đảm bảo khả năng tương thích 100% với các mã kiểm định giả thuyết (`hypothesis_tests_P.py`); chỉ tạo thêm các cột đặc trưng phái sinh.

---

### Sơ đồ quy trình 11 bước:
```
1 Đọc dữ liệu ➔ 2 Kiểm tra Schema ➔ 3 Lọc phạm vi WA ➔ 4 Chuẩn hóa văn bản ➔ 5 Ép kiểu dữ liệu
➔ 6 Xử lý bất hợp lệ ➔ 7 Tạo đặc trưng ➔ 8 Kiểm toán thiếu ➔ 9 Kiểm toán trùng ➔ 10 Đối chiếu báo cáo ➔ 11 Lưu trữ
```


## Thiết lập môi trường và tham số ban đầu

In [1]:
from __future__ import annotations

import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

# Tránh lỗi mã hóa tiếng Việt trên console Windows
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# Xác định thư mục gốc của repository
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
DEFAULT_IN = ROOT / "data" / "EV_Population_Washington_Clean__1_.csv"
DEFAULT_OUT = ROOT / "data" / "processed" / "EV_Population_WA_Processed.csv"
AUDIT_OUT = ROOT / "results" / "P_preprocessing_audit.csv"

# ----------------------------------------------------------------------------- Tên các cột
COUNTY, CITY, STATE, ZIP = "County", "City", "State", "Postal Code"
YEAR, MAKE, MODEL = "Model Year", "Make", "Model"
TYPE_RAW = "Electric Vehicle Type"
CAFV = "Clean Alternative Fuel Vehicle (CAFV) Eligibility"
RANGE, MSRP = "Electric Range", "Base MSRP"

EXPECTED_COLUMNS = [COUNTY, CITY, STATE, ZIP, YEAR, MAKE, MODEL, TYPE_RAW, CAFV, RANGE, MSRP]
TEXT_COLUMNS = [COUNTY, CITY, STATE, MAKE, MODEL, TYPE_RAW, CAFV]

# ----------------------------------------------------------------------------- Quy tắc nghiệp vụ
STUDY_STATE = "WA"
YEAR_MIN, YEAR_MAX = 1980, date.today().year + 1   # Khoảng năm sản xuất hợp lý
RANGE_MAX_MILES = 500                               # Giới hạn tầm chạy pin tối đa cho xe thương mại
WA_ZIP_MIN, WA_ZIP_MAX = 98001, 99403               # Dải mã bưu chính (ZIP code) của bang Washington
CITY_OVERRIDES = {"MCCLEARY": "McCleary"}           # Chuẩn hóa biến thể viết hoa/thường đặc thù
CAFV_SHORT = {
    "Clean Alternative Fuel Vehicle Eligible": "Eligible",
    "Not eligible due to low battery range": "Not eligible",
    "Eligibility unknown as battery range has not been researched": "Unknown",
}

# Các số liệu công bố trong báo cáo Nhóm 8 - dùng để kiểm định tự động tính toàn vẹn
REPORT_FIGURES = {
    "Số bản ghi (N)": 134_726,
    "Số xe BEV": 103_674,
    "Số xe PHEV": 31_052,
    "Số County phân biệt": 39,
    "Số Make (Hãng xe) phân biệt": 36,
    "Số Model (không rỗng)": 125,
    "Số Model Year phân biệt": 22,
    "Số dòng thiếu Electric Range": 56_869,
    "Số dòng có Base MSRP": 3_419,
    "Số dòng thiếu Model": 249,
    "Số xe nhãn CAFV 'Unknown'": 56_868,
}

AUDIT: list[dict] = []   # Nhật ký kiểm toán các bước xử lý


def log(step: str, rows_in: int, rows_out: int, detail: str) -> None:
    """Ghi nhận hành động vào danh sách kiểm toán và in tóm tắt ra màn hình."""
    AUDIT.append({"Bước": step, "Số dòng vào": rows_in, "Số dòng ra": rows_out, "Chi tiết thực hiện": detail})
    print(f"[{step}] Dòng: {rows_in:,} -> {rows_out:,} | {detail}")

## Bước 1: Đọc dữ liệu thô (Load)

- **What (Làm gì?):** Đọc file CSV vào DataFrame mà không để pandas tự suy đoán sai kiểu dữ liệu của cột mã bưu chính (`Postal Code`).
- **How (Làm như thế nào?):** Sử dụng `pd.read_csv(path, dtype={'Postal Code': 'string'})`; các cột còn lại để pandas tự tải theo mặc định.
- **Why (Tại sao phải làm?):** File CSV lưu mã ZIP dưới dạng số thực float (`99344.0`). Nếu để pandas tự đọc thành số, mã ZIP có thể bị mất số 0 ở đầu hoặc biến thành số học; ép kiểu chuỗi text ngay từ đầu giúp bảo toàn dữ liệu gốc cho bước chuẩn hóa phía sau.

In [2]:
def load_raw(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={ZIP: "string"})
    log("1 Đọc dữ liệu", len(df), len(df), f"Đã nạp {df.shape[1]} cột từ file {path.name}")
    return df

In [3]:
df = load_raw(DEFAULT_IN)
df.head()

[1 Đọc dữ liệu] Dòng: 134,726 -> 134,726 | Đã nạp 11 cột từ file EV_Population_Washington_Clean__1_.csv


,County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Base MSRP
0,Adams,Hatton,WA,99344.0,2020.0,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,266.0,NaN
1,Adams,Lind,WA,99341.0,2020.0,CHRYSLER,PACIFICA,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,32.0,NaN
2,Adams,Lind,WA,99341.0,2022.0,RIVIAN,R1T,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,NaN,NaN
3,Adams,Othello,WA,99344.0,2011.0,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,73.0,NaN
4,Adams,Othello,WA,99344.0,2012.0,CHEVROLET,VOLT,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,35.0,NaN


## Bước 2: Kiểm tra cấu trúc bảng (Schema Validation)

- **What (Làm gì?):** Xác minh chính xác 11 cột thuộc tính như đã mô tả trong báo cáo có mặt đầy đủ trong dữ liệu hay không.
- **How (Làm như thế nào?):** So sánh tập hợp tên cột với `EXPECTED_COLUMNS`; kích hoạt `ValueError` nếu thiếu cột và loại bỏ các cột lạ ngoài danh mục.
- **Why (Tại sao phải làm?):** Tất cả các hàm phân tích tiếp theo đều truy xuất cột theo tên cố định. Phát hiện lỗi sớm (fail-fast) giúp tránh việc phân tích trên file sai cấu trúc; việc giữ đúng 11 cột đảm bảo dữ liệu chuẩn hóa đúng theo Bảng 2.1 và 3.1 của báo cáo.

In [4]:
def validate_schema(df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Thiếu các cột bắt buộc: {missing}")
    extra = [c for c in df.columns if c not in EXPECTED_COLUMNS]
    df = df[EXPECTED_COLUMNS].copy()
    log("2 Kiểm tra Schema", len(df), len(df), f"11 cột hợp lệ; Cột dư thừa đã loại: {extra or 'Không có'}")
    return df

In [5]:
df = validate_schema(df)

[2 Kiểm tra Schema] Dòng: 134,726 -> 134,726 | 11 cột hợp lệ; Cột dư thừa đã loại: Không có


## Bước 3: Lọc phạm vi nghiên cứu (State = WA)

- **What (Làm gì?):** Chỉ giữ lại các phương tiện có địa bàn đăng ký thuộc Bang Washington (`State == 'WA'`).
- **How (Làm như thế nào?):** Tạo mặt nạ boolean kiểm tra `State.str.strip().str.upper() == 'WA'` và lọc dữ liệu.
- **Why (Tại sao phải làm?):** Theo phạm vi nghiên cứu (Mục 1.4 & 3.2), đề tài chỉ khảo sát xe tại bang Washington. Bước này loại bỏ các xe đăng ký ngoài bang (nếu có) và đóng vai trò như chốt an toàn khi chạy trên file thô 17 cột.

In [6]:
def filter_scope(df: pd.DataFrame) -> pd.DataFrame:
    n0 = len(df)
    mask = df[STATE].astype("string").str.strip().str.upper() == STUDY_STATE
    df = df[mask].copy()
    log("3 Lọc phạm vi", n0, len(df), f"Đã loại {n0 - len(df):,} dòng có State != {STUDY_STATE}")
    return df

In [7]:
df = filter_scope(df)

[3 Lọc phạm vi] Dòng: 134,726 -> 134,726 | Đã loại 0 dòng có State != WA


## Bước 4: Chuẩn hóa chuỗi văn bản (Text Standardisation)

- **What (Làm gì?):** Loại bỏ khoảng trắng thừa đầu/cuối, gom cụm nhiều khoảng trắng liên tiếp, viết hoa đồng nhất hãng/dòng xe và chuẩn hóa tên thành phố.
- **How (Làm như thế nào?):** (a) Cắt tỉa và chuẩn hóa khoảng trắng trên tất cả các cột text;
(b) Viết hoa toàn bộ `Make` và `Model`;
(c) Đồng nhất chữ hoa/thường cho `City` (như biến thể `Mccleary` thành `McCleary`);
(d) Kiểm tra tính hợp lệ của nhãn `Electric Vehicle Type` và `CAFV Eligibility`.
- **Why (Tại sao phải làm?):** Nếu không chuẩn hóa, 'TESLA' và 'Tesla', hay 'Mccleary' và 'McCleary' sẽ bị đếm thành các nhóm riêng biệt, làm sai lệch kết quả Top-N và bảng chéo kiểm định $\chi^2$ (H3, H4a). Lưu ý: Tên County được giữ nguyên kiểu Title Case vì đã đồng nhất 39 hạt từ trước.

In [8]:
def standardize_text(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    changed = {}
    for c in TEXT_COLUMNS:
        before = out[c].astype("string")
        after = before.str.strip().str.replace(r"\s+", " ", regex=True)
        if c in (MAKE, MODEL):
            after = after.str.upper()
        changed[c] = int((before.fillna("") != after.fillna("")).sum())
        out[c] = after

    # Đồng nhất biến thể viết hoa/thường của tên City
    key = out[CITY].str.upper()
    counts = (pd.DataFrame({"key": key, "variant": out[CITY]})
              .value_counts().rename("n").reset_index())
    canon = counts.sort_values(["key", "n"], ascending=[True, False]).drop_duplicates("key")
    canon_map = dict(zip(canon["key"], canon["variant"])) | CITY_OVERRIDES
    new_city = key.map(canon_map)
    changed[CITY] += int((out[CITY] != new_city).sum())
    out[CITY] = new_city

    # Kiểm tra tính hợp lệ của nhãn loại xe và CAFV
    bad_type = ~out[TYPE_RAW].str.extract(r"\((BEV|PHEV)\)")[0].notna()
    bad_cafv = ~out[CAFV].isin(CAFV_SHORT)
    if bad_type.any() or bad_cafv.any():
        raise ValueError(f"Phát hiện nhãn bất thường: {bad_type.sum()} loại xe, {bad_cafv.sum()} CAFV")

    log("4 Chuẩn hóa văn bản", len(df), len(out),
        "Số ô được sửa đổi: " + (", ".join(f"{k}={v}" for k, v in changed.items() if v) or "Không có"))
    log("4 Chuẩn hóa văn bản", len(out), len(out),
        f"Số City phân biệt: {df[CITY].nunique()} -> {out[CITY].nunique()}; "
        f"Make: {out[MAKE].nunique()}; Model: {out[MODEL].nunique()}")
    return out

In [9]:
df = standardize_text(df)

[4 Chuẩn hóa văn bản] Dòng: 134,726 -> 134,726 | Số ô được sửa đổi: City=21
[4 Chuẩn hóa văn bản] Dòng: 134,726 -> 134,726 | Số City phân biệt: 454 -> 453; Make: 36; Model: 125


## Bước 5: Ép kiểu dữ liệu chuẩn (Type Casting)

- **What (Làm gì?):** Gán đúng kiểu dữ liệu thực tế cho từng biến số: `Postal Code` thành chuỗi 5 ký tự; `Model Year`, `Electric Range`, `Base MSRP` thành kiểu số nguyên cho phép rỗng (`Int64`).
- **How (Làm như thế nào?):** Dùng `pd.to_numeric` $\to$ `.astype('Int64')`. Với mã ZIP dùng `.str.zfill(5)` để bù đủ 5 chữ số.
- **Why (Tại sao phải làm?):** CSV lưu số nguyên dưới dạng float (`2020.0`, `266.0`). Nếu để float, năm sản xuất sẽ hiển thị là 2020.0 gây khó đọc và sai logic; mã ZIP là biến định danh chứ không phải đại lượng toán học. Sử dụng nullable `Int64` (chữ I viết hoa của pandas) để bảo toàn các ô thiếu (NA) mà không bị tự ý đổi thành số 0.

In [10]:
def cast_types(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    zip_num = pd.to_numeric(out[ZIP], errors="coerce").astype("Int64")
    out[ZIP] = zip_num.astype("string").str.zfill(5)
    for c in (YEAR, RANGE, MSRP):
        num = pd.to_numeric(out[c], errors="coerce")
        non_int = int(((num % 1 != 0) & num.notna()).sum())
        if non_int:
            raise ValueError(f"Cột {c}: có {non_int} giá trị không phải số nguyên; từ chối làm tròn")
        out[c] = num.astype("Int64")
    log("5 Ép kiểu dữ liệu", len(df), len(out), "Postal Code -> str(5); Model Year / Electric Range / Base MSRP -> Int64")
    return out

In [11]:
df = cast_types(df)
df.dtypes

[5 Ép kiểu dữ liệu] Dòng: 134,726 -> 134,726 | Postal Code -> str(5); Model Year / Electric Range / Base MSRP -> Int64


County                                               string
City                                                    str
State                                                string
Postal Code                                          string
Model Year                                            Int64
Make                                                 string
Model                                                string
Electric Vehicle Type                                string
Clean Alternative Fuel Vehicle (CAFV) Eligibility    string
Electric Range                                        Int64
Base MSRP                                             Int64
dtype: object

## Bước 6: Xử lý giá trị bất hợp lệ và cảnh báo bất thường

- **What (Làm gì?):** Chuyển đổi các giá trị vô lý hoặc mã placeholder thành NA: Tầm chạy pin $\le 0$ hoặc $> 500$ dặm; Giá MSRP $\le 0$; Năm sản xuất ngoài khoảng 1980..nay. Đồng thời rà soát mã bưu chính bất thường.
- **How (Làm như thế nào?):** Sử dụng mặt nạ logic và phương thức `.mask()` để gán NA cho các ô vi phạm; ghi log kiểm toán.
- **Why (Tại sao phải làm?):** Cơ quan DOL dùng giá trị 0 cho các mẫu xe chưa nghiên cứu tầm pin. Nếu giữ nguyên số 0, trung bình và các phép kiểm định phi tham số (H1, H2) sẽ bị kéo lệch nghiêm trọng. Giữ NA giúp các phép tính chỉ thực hiện trên quan sát hợp lệ (valid cases). Các xe đời trước 2010 (như Tesla Roadster 2008, Ford Ranger EV 1998) là xe có thật trong lịch sử nên được giữ nguyên.

In [12]:
def handle_invalid_values(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    r_bad = (out[RANGE] <= 0) | (out[RANGE] > RANGE_MAX_MILES)
    m_bad = out[MSRP] <= 0
    y_bad = (out[YEAR] < YEAR_MIN) | (out[YEAR] > YEAR_MAX)
    out[RANGE] = out[RANGE].mask(r_bad.fillna(False))
    out[MSRP] = out[MSRP].mask(m_bad.fillna(False))
    out[YEAR] = out[YEAR].mask(y_bad.fillna(False))
    zip_int = pd.to_numeric(out[ZIP], errors="coerce")
    zip_out = int((~zip_int.between(WA_ZIP_MIN, WA_ZIP_MAX) | out[ZIP].str.len().ne(5)).sum())
    old = int((out[YEAR] < 2010).sum())
    log("6 Xử lý bất hợp lệ", len(df), len(out),
        f"Range -> NA: {int(r_bad.sum())}; MSRP -> NA: {int(m_bad.sum())}; Year -> NA: {int(y_bad.sum())}; "
        f"ZIP ngoài dải WA: {zip_out}; Số xe đời < 2010 giữ nguyên: {old}")
    return out

In [13]:
df = handle_invalid_values(df)

[6 Xử lý bất hợp lệ] Dòng: 134,726 -> 134,726 | Range -> NA: 0; MSRP -> NA: 0; Year -> NA: 0; ZIP ngoài dải WA: 0; Số xe đời < 2010 giữ nguyên: 37


## Bước 7: Kỹ thuật tạo đặc trưng (Feature Engineering)

- **What (Làm gì?):** Tạo thêm 4 biến dẫn xuất hữu ích phục vụ trực tiếp cho mô tả và kiểm định (không làm thay đổi các cột gốc).
- **How (Làm như thế nào?):** (a) `Type`: Trích xuất nhãn ngắn 'BEV' hoặc 'PHEV';
(b) `CAFV Status`: Rút gọn nhãn CAFV thành 'Eligible', 'Not eligible', 'Unknown';
(c) `Range Missing`: Biến cờ boolean nhận diện dòng thiếu tầm pin;
(d) `County City`: Khóa kết hợp dạng `<County> | <City>`.
- **Why (Tại sao phải làm?):** - `Type` là biến phân nhóm trung tâm của H1, H2, H3, H4a; việc chuẩn hóa sẵn giúp các mã phân tích sau không phải viết lại logic regex.
- `County City`: Có 38 tên thành phố trùng tên nằm ở các hạt khác nhau (tổng cộng 493 cặp City-County); việc tạo khóa kép giúp tránh gộp nhầm các địa phương khác nhau khi tổng hợp.

In [14]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Type"] = out[TYPE_RAW].str.extract(r"\((BEV|PHEV)\)")[0]
    out["CAFV Status"] = out[CAFV].map(CAFV_SHORT)
    out["Range Missing"] = out[RANGE].isna()
    out["County City"] = out[COUNTY] + " | " + out[CITY]
    log("7 Tạo đặc trưng", len(df), len(out), "Đã thêm 4 cột mới: Type, CAFV Status, Range Missing, County City")
    return out

In [15]:
df = engineer_features(df)
df[['Type', 'CAFV Status', 'Range Missing', 'County City']].head()

[7 Tạo đặc trưng] Dòng: 134,726 -> 134,726 | Đã thêm 4 cột mới: Type, CAFV Status, Range Missing, County City


,Type,CAFV Status,Range Missing,County City
0,BEV,Eligible,False,Adams | Hatton
1,PHEV,Eligible,False,Adams | Lind
2,BEV,Unknown,True,Adams | Lind
3,BEV,Eligible,False,Adams | Othello
4,PHEV,Eligible,False,Adams | Othello


## Bước 8: Đánh giá và kiểm toán giá trị thiếu (Missing Data Audit)

- **What (Làm gì?):** Đo lường chi tiết tỷ lệ thiếu của từng biến, so sánh theo loại xe BEV/PHEV và đối chiếu với nhãn CAFV 'Unknown'. Tuyệt đối **không điền khuyết (no imputation)**.
- **How (Làm như thế nào?):** Dùng `.isna().sum()`, `groupby('Type')` và lập bảng chéo kiểm tra mức độ trùng khớp giữa `Range Missing` và `CAFV Status == 'Unknown'`.
- **Why (Tại sao phải làm?):** Dữ liệu thiếu ở đây có tính chất phi ngẫu nhiên (MNAR / structural): 54,85% xe BEV bị thiếu tầm pin (chủ yếu từ đời 2021 trở đi do DOL chưa kịp cập nhật kiểm định pin), trong khi PHEV chỉ thiếu 1 xe duy nhất. Điền khuyết (bằng trung bình/trung vị hay 0) sẽ tạo ra dữ liệu giả mạo làm sai lệch hoàn toàn kiểm định H1 và H2. Quy tắc chuẩn: giữ nguyên NA, kiểm định trên tập mẫu hợp lệ và ghi rõ cỡ mẫu N tương ứng.

In [16]:
def audit_missing(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    miss = df[EXPECTED_COLUMNS].isna().sum()
    miss = miss[miss > 0]
    by_type = df.groupby("Type")["Range Missing"].agg(["sum", "size"])
    unk = df["CAFV Status"].eq("Unknown")
    both = int((unk & df["Range Missing"]).sum())
    log("8 Kiểm toán thiếu", n, n,
        "; ".join(f"{c}={v:,} ({v / n:.2%})" for c, v in miss.items()))
    log("8 Kiểm toán thiếu", n, n,
        "Thiếu Range theo loại xe: " + ", ".join(f"{t}={int(r['sum']):,}/{int(r['size']):,}"
                                                  for t, r in by_type.iterrows())
        + f"; Trùng khớp nhãn CAFV Unknown & Range thiếu: {both:,}/{int(unk.sum()):,}")
    return df

In [17]:
df = audit_missing(df)

[8 Kiểm toán thiếu] Dòng: 134,726 -> 134,726 | Model=249 (0.18%); Electric Range=56,869 (42.21%); Base MSRP=131,307 (97.46%)
[8 Kiểm toán thiếu] Dòng: 134,726 -> 134,726 | Thiếu Range theo loại xe: BEV=56,868/103,674, PHEV=1/31,052; Trùng khớp nhãn CAFV Unknown & Range thiếu: 56,868/56,868


## Bước 9: Đánh giá bản ghi trùng lặp (Duplicate Audit)

- **What (Làm gì?):** Đếm số lượng dòng trùng lặp hoàn toàn trên toàn bộ 11 cột thuộc tính ban đầu. **Tuyệt đối không xóa bỏ dòng trùng**.
- **How (Làm như thế nào?):** Sử dụng hàm `.duplicated(subset=EXPECTED_COLUMNS)` và tính kích thước nhóm trùng lớn nhất.
- **Why (Tại sao phải làm?):** File dữ liệu sạch không chứa mã số định danh xe (`DOL Vehicle ID`). Các xe cùng hãng, cùng dòng, cùng năm sản xuất, đăng ký tại cùng một thành phố và mã ZIP (ví dụ: các xe Tesla Model Y 2022 tại Seattle) sẽ có các thuộc tính giống hệt nhau. Nếu tùy tiện xóa dòng trùng, chúng ta sẽ xóa mất gần 70% số lượng xe thực tế, phá vỡ toàn bộ tỷ trọng cơ cấu BEV/PHEV và phân bố địa lý.

In [18]:
def audit_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    dup = int(df.duplicated(subset=EXPECTED_COLUMNS).sum())
    biggest = int(df.groupby(EXPECTED_COLUMNS, dropna=False).size().max())
    log("9 Kiểm toán trùng lặp", n, n,
        f"{dup:,} dòng ({dup / n:.1%}) trùng lặp trên 11 cột; Nhóm trùng lớn nhất: {biggest:,} xe; "
        "KHÔNG XÓA DÒNG (mỗi dòng là một phương tiện thực tế)")
    return df

In [19]:
df = audit_duplicates(df)

[9 Kiểm toán trùng lặp] Dòng: 134,726 -> 134,726 | 93,707 dòng (69.6%) trùng lặp trên 11 cột; Nhóm trùng lớn nhất: 562 xe; KHÔNG XÓA DÒNG (mỗi dòng là một phương tiện thực tế)


## Bước 10: Đối chiếu tự động với số liệu Báo cáo Nhóm 8

- **What (Làm gì?):** Tái tính toán 11 chỉ số cơ bản của tập dữ liệu sau xử lý và đối chiếu trực tiếp với các số liệu đã công bố trong bài báo cáo.
- **How (Làm như thế nào?):** Tính toán thực tế các chỉ số (tổng dòng, số BEV/PHEV, số hạt, số hãng, số ô thiếu...) và so sánh với từ điển `REPORT_FIGURES`.
- **Why (Tại sao phải làm?):** Đây là bước kiểm thử hồi quy (regression test) bắt buộc nhằm đảm bảo đường ống tiền xử lý không làm thất thoát hoặc biến dạng bất kỳ quan sát nào, duy trì tính tái lập (reproducibility) 100% cho toàn bộ công trình nghiên cứu.

In [20]:
def validate_against_report(df: pd.DataFrame, strict: bool = False) -> None:
    actual = {
        "Số bản ghi (N)": len(df),
        "Số xe BEV": int((df["Type"] == "BEV").sum()),
        "Số xe PHEV": int((df["Type"] == "PHEV").sum()),
        "Số County phân biệt": df[COUNTY].nunique(),
        "Số Make (Hãng xe) phân biệt": df[MAKE].nunique(),
        "Số Model (không rỗng)": df[MODEL].nunique(),
        "Số Model Year phân biệt": df[YEAR].nunique(),
        "Số dòng thiếu Electric Range": int(df[RANGE].isna().sum()),
        "Số dòng có Base MSRP": int(df[MSRP].notna().sum()),
        "Số dòng thiếu Model": int(df[MODEL].isna().sum()),
        "Số xe nhãn CAFV 'Unknown'": int((df["CAFV Status"] == "Unknown").sum()),
    }
    diffs = 0
    print("=== KẾT QUẢ ĐỐI CHIẾU SỐ LIỆU VỚI BÁO CÁO ===")
    for k, expected in REPORT_FIGURES.items():
        ok = actual[k] == expected
        diffs += not ok
        print(f"   [{'KHỚP ' if ok else 'LỆCH'}] {k:32s} | Dữ liệu: {actual[k]:>8,} | Báo cáo: {expected:>8,}")
    n_pairs = df[["County", "City"]].drop_duplicates().shape[0]
    print(f"\n   [THÔNG TIN] Số City sau chuẩn hóa = {df[CITY].nunique()} "
          f"(Báo cáo ghi 454 tên chuỗi thô) | Số cặp (County, City) = {n_pairs}")
    log("10 Đối chiếu báo cáo", len(df), len(df), f"Tái lập chính xác {len(REPORT_FIGURES) - diffs}/{len(REPORT_FIGURES)} chỉ số báo cáo")
    if strict and diffs:
        raise AssertionError(f"Có {diffs} chỉ số sai lệch so với báo cáo!")

In [21]:
validate_against_report(df)

=== KẾT QUẢ ĐỐI CHIẾU SỐ LIỆU VỚI BÁO CÁO ===
   [KHỚP ] Số bản ghi (N)                   | Dữ liệu:  134,726 | Báo cáo:  134,726
   [KHỚP ] Số xe BEV                        | Dữ liệu:  103,674 | Báo cáo:  103,674
   [KHỚP ] Số xe PHEV                       | Dữ liệu:   31,052 | Báo cáo:   31,052
   [KHỚP ] Số County phân biệt              | Dữ liệu:       39 | Báo cáo:       39
   [KHỚP ] Số Make (Hãng xe) phân biệt      | Dữ liệu:       36 | Báo cáo:       36
   [KHỚP ] Số Model (không rỗng)            | Dữ liệu:      125 | Báo cáo:      125
   [KHỚP ] Số Model Year phân biệt          | Dữ liệu:       22 | Báo cáo:       22
   [KHỚP ] Số dòng thiếu Electric Range     | Dữ liệu:   56,869 | Báo cáo:   56,869
   [KHỚP ] Số dòng có Base MSRP             | Dữ liệu:    3,419 | Báo cáo:    3,419
   [KHỚP ] Số dòng thiếu Model              | Dữ liệu:      249 | Báo cáo:      249
   [KHỚP ] Số xe nhãn CAFV 'Unknown'        | Dữ liệu:   56,868 | Báo cáo:   56,868

   [THÔNG TIN] Số City sau ch

## Bước 11: Lưu trữ dữ liệu sạch và xuất nhật ký kiểm toán

- **What (Làm gì?):** Xuất file dữ liệu đã tiền xử lý hoàn chỉnh ra thư mục `data/processed/` và lưu bảng kiểm toán các bước vào `results/`.
- **How (Làm như thế nào?):** Sử dụng `df.to_csv(..., encoding='utf-8-sig')` để hỗ trợ hiển thị tiếng Việt hoàn hảo trên Excel.
- **Why (Tại sao phải làm?):** Cung cấp một file dữ liệu duy nhất, nhất quán cho toàn bộ các sổ tay phân tích EDA, trực quan hóa và kiểm định giả thuyết (H1-H4) phía sau; tránh việc mỗi thành viên phải chạy lại khâu làm sạch.

In [22]:
def save(df: pd.DataFrame, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    AUDIT_OUT.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    pd.DataFrame(AUDIT).to_csv(AUDIT_OUT, index=False, encoding="utf-8-sig")
    print(f"\nĐã lưu thành công {len(df):,} dòng x {df.shape[1]} cột vào: {out_path.name}")
    print(f"Đã lưu nhật ký kiểm toán vào: {AUDIT_OUT.name}")

In [23]:
save(df, DEFAULT_OUT)


Đã lưu thành công 134,726 dòng x 15 cột vào: EV_Population_WA_Processed.csv
Đã lưu nhật ký kiểm toán vào: P_preprocessing_audit.csv


## Tổng kết nhật ký kiểm toán (Audit Trail)

Bảng dưới đây thống kê số lượng dòng vào/ra và các chi tiết can thiệp qua từng bước xử lý. Bảng này cũng được lưu tự động tại `results/P_preprocessing_audit.csv`.

In [24]:
pd.DataFrame(AUDIT)

,Bước,Số dòng vào,Số dòng ra,Chi tiết thực hiện
0,1 Đọc dữ liệu,134726,134726,Đã nạp 11 cột từ file EV_Population_Washington...
1,2 Kiểm tra Schema,134726,134726,11 cột hợp lệ; Cột dư thừa đã loại: Không có
2,3 Lọc phạm vi,134726,134726,Đã loại 0 dòng có State != WA
3,4 Chuẩn hóa văn bản,134726,134726,Số ô được sửa đổi: City=21
4,4 Chuẩn hóa văn bản,134726,134726,Số City phân biệt: 454 -> 453; Make: 36; Model...
5,5 Ép kiểu dữ liệu,134726,134726,Postal Code -> str(5); Model Year / Electric R...
6,6 Xử lý bất hợp lệ,134726,134726,Range -> NA: 0; MSRP -> NA: 0; Year -> NA: 0; ...
7,7 Tạo đặc trưng,134726,134726,"Đã thêm 4 cột mới: Type, CAFV Status, Range Mi..."
8,8 Kiểm toán thiếu,134726,134726,"Model=249 (0.18%); Electric Range=56,869 (42.2..."
9,8 Kiểm toán thiếu,134726,134726,"Thiếu Range theo loại xe: BEV=56,868/103,674, ..."


## Các phát hiện quan trọng phục vụ hoàn thiện Báo cáo

1. **Về số lượng Thành phố (City):**
   - Trong báo cáo gốc ghi nhận **454** City theo tên chuỗi thô.
   - Sau khi chuẩn hóa chữ hoa/thường (ví dụ `Mccleary` và `McCleary`), số City thực tế là **453**. Số cặp `(County, City)` vẫn bảo toàn là **493**.
   - *Khuyến nghị:* Nhóm có thể ghi rõ trong báo cáo: *"454 tên chuỗi thô, tương ứng 453 thành phố chuẩn hóa"*.

2. **Về Thành phố McCleary:**
   - Xuất hiện ở 2 County: Mason (1 xe) và Grays Harbor (21 xe). Đây là hiện tượng nhập liệu thực tế từ cơ sở dữ liệu DOL của bang, quy trình giữ nguyên tính khách quan.

3. **Giải đáp các Comment phản biện C1, C2, C3 trong bản thảo báo cáo:**
   - **C1:** Toàn bộ 134.726 dòng trong file sạch đều có `State == WA` (0 dòng bị loại).
   - **C2:** Cột `Electric Range` và `Base MSRP` trong file sạch không chứa giá trị `0` (đã ở dạng thiếu `NA` từ trước).
   - **C3:** Do file sạch đã lược bỏ cột `DOL Vehicle ID`, toàn bộ 134.726 dòng được giữ nguyên vẹn để phản ánh đúng quy mô đội xe.